# Generating and evaluating a landing page brief

**Time**: ~35-45 minutes, most of it waiting on model calls. **Cost**: a few cents at most on Gemini's cheapest models -- this notebook makes on the order of 50-60 small calls, most of them the evaluation section's judging calls. If you haven't run [Session 1's setup guide](../week_2/week2_lesson08_setup_guide.ipynb) yet, do that first.

> **Running this locally, or in VS Code instead of Colab?** See [Session 1's setup guide](../week_2/week2_lesson08_setup_guide.ipynb) for how to open any of these notebooks with `uv`, either in a browser tab or inside VS Code. Nothing extra to do if you're in Colab.

[`agentic_content_pipeline.ipynb`](agentic_content_pipeline.ipynb) built a gap table and an agent that reads it through three tools to produce a short recommendation. This notebook goes one level deeper on a single output type: a full **landing page brief**, the kind a content team would actually work from, generated for the single highest-demand gap. Writing the brief is the easy half. The harder half, and the one this notebook spends most of its time on, is checking whether the brief is any good -- and specifically, whether it's good in a way that traces back to real data rather than to the model's own confidence.

**What you'll do:**
1. Rebuild the gap table and pick the single highest-demand gap to brief.
2. Define what a landing page brief needs to contain before generating one -- the same "write the taxonomy down first" discipline this week's overview described for intent labeling.
3. Generate a brief, with a guardrail that refuses to brief a cluster that isn't actually flagged as a gap.
4. Score the brief by reusing Session 2's evaluation template -- reframed as a set of pass/fail checks, most of them checkable against the real underlying data without a human reading anything.
5. Identify which part of the pipeline -- the upstream gap table, or the brief-writing step itself -- the errors actually trace back to.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab: installs from an explicit list, not week_7/requirements.txt --
    # "Open in Colab" only loads this one file, so there's no repo alongside it
    # to read that file from. This list is that file's contents written out
    # directly; keep the two in sync if you add or remove a package.
    %pip install -q google-genai pandas numpy matplotlib seaborn scikit-learn
else:
    # Local: installs from week_7/requirements.txt. Looked up by directory,
    # not just filename, since the repo also has its own top-level
    # requirements.txt for whole-repo setup -- a same-named but wrong file a
    # plain filename search could grab by mistake from the repo root.
    import pathlib
    _cwd = pathlib.Path.cwd()
    if _cwd.name == "week_7" and (_cwd / "requirements.txt").exists():
        _req_path = "requirements.txt"
    elif (_cwd / "week_7" / "requirements.txt").exists():
        _req_path = "week_7/requirements.txt"
    elif (_cwd.parent / "week_7" / "requirements.txt").exists():
        _req_path = "../week_7/requirements.txt"
    else:
        _req_path = None

    if _req_path is None:
        print("Couldn't find week_7/requirements.txt from the current working directory:", _cwd)
        print("Set up your local environment first -- see Session 1's setup guide for the uv commands.")
    else:
        %pip install -q -r {_req_path}

### A note on privacy

Same rule as every other session: work only with public example data, synthetic data, or the anonymized examples this course provides. Everything this notebook runs on -- Larkspur Trail Co., its pages, its search performance export -- is invented for this exercise. When you point this pipeline at a real property later, remember that unpaid or free-quota API usage may be reviewed by humans or used to improve models; billed usage on a Gemini "Paid tier" project is not, but confirm that's the tier your key is actually on at [aistudio.google.com](https://aistudio.google.com) rather than assuming it.

In [ ]:
from google import genai
from google.colab import userdata

MODEL_NAME = "gemini-3.5-flash-lite"
EMBEDDING_MODEL_NAME = "gemini-embedding-001"

client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))


def call_llm(prompt: str, model: str = None) -> str:
    response = client.models.generate_content(model=model or MODEL_NAME, contents=prompt)
    return response.text


def get_embeddings(texts: list) -> list:
    result = client.models.embed_content(model=EMBEDDING_MODEL_NAME, contents=texts)
    return [e.values for e in result.embeddings]


print("Connected. Model:", MODEL_NAME)

## Reconnect: rebuilding the gap table

Same method as [`agentic_content_pipeline.ipynb`](agentic_content_pipeline.ipynb), condensed into one function so this notebook can run on its own -- embed the queries, cluster with HDBSCAN, label and classify each cluster against the written taxonomy, embed the pages, score by cosine similarity, and set a threshold. If any of that reads as unfamiliar, that notebook is where every step is explained and tried at several settings before being picked.

In [ ]:
import pandas as pd
import numpy as np
import pathlib
from sklearn.cluster import HDBSCAN
from sklearn.preprocessing import normalize

DATA_DIR_CANDIDATES = [
    pathlib.Path("data/search_console"),
    pathlib.Path("week_7/data/search_console"),
    pathlib.Path("../week_7/data/search_console"),
]
RAW_BASE = "https://raw.githubusercontent.com/xkpacx/AI4TM/main/week_7/data/search_console"


def load_csv(filename: str) -> pd.DataFrame:
    for d in DATA_DIR_CANDIDATES:
        path = d / filename
        if path.exists():
            return pd.read_csv(path)
    return pd.read_csv(f"{RAW_BASE}/{filename}")


QUESTION_WORDS = {"how", "what", "when", "where", "why", "which", "who", "whose", "can", "do", "does", "is", "are"}
INTENT_TAXONOMY = """
- knowledge-seeking: the goal is to understand or find out something
- guidance-seeking: the goal is help with a decision or a course of action
- output-seeking: the goal is an artefact the system produces, such as text, code, or a plan
- navigational: the goal is to reach a specific known page or account
- transactional: the goal is to complete a purchase or similar action
"""
ALLOWED_INTENTS = {"knowledge-seeking", "guidance-seeking", "output-seeking", "navigational", "transactional", "ambiguous"}


def label_cluster(cluster_queries: list) -> str:
    prompt = f"""These are real search queries a website received:
{chr(10).join('- ' + q for q in cluster_queries)}

State the single underlying question these queries share, as a short, natural phrase (the
kind you'd use as a page title). Answer with only that phrase, nothing else."""
    return call_llm(prompt).strip().strip('"')


def classify_intent(cluster_queries: list) -> str:
    prompt = f"""Taxonomy of search intent:
{INTENT_TAXONOMY}

Real search queries:
{chr(10).join('- ' + q for q in cluster_queries)}

Which single category from the taxonomy best describes what these people are trying to do?
Answer with exactly one of: knowledge-seeking, guidance-seeking, output-seeking, navigational,
transactional, ambiguous. Use "ambiguous" only if the queries genuinely split across categories
with no majority. Answer with only that one word, nothing else."""
    label = call_llm(prompt).strip().lower().strip(".")
    return label if label in ALLOWED_INTENTS else "ambiguous"


def cosine_similarity(a, b) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def build_gap_table(perf_df: pd.DataFrame, pages_df: pd.DataFrame, min_cluster_size: int = 2) -> pd.DataFrame:
    """Condensed rebuild of the pipeline from agentic_content_pipeline.ipynb. Uses a
    placeholder median-score threshold rather than that notebook's human-labeled procedure --
    see it for the real three-step method with an actual human read of each pair."""
    all_queries = perf_df["Query"].tolist()
    query_vectors = np.array(get_embeddings(all_queries))
    query_vectors_normalized = normalize(query_vectors)
    cluster_ids = HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean").fit_predict(query_vectors_normalized)

    work = perf_df.copy()
    work["cluster_id"] = cluster_ids

    clusters = []
    for cid, group in work[work["cluster_id"] != -1].groupby("cluster_id"):
        cluster_queries = group["Query"].tolist()
        clusters.append({
            "cluster_id": int(cid),
            "queries": cluster_queries,
            "impressions": int(group["Impressions"].sum()),
            "clicks": int(group["Clicks"].sum()),
            "mean_word_count": round(sum(len(q.split()) for q in cluster_queries) / len(cluster_queries), 1),
            "pct_question_form": round(sum(q.strip().split()[0].lower() in QUESTION_WORDS for q in cluster_queries) / len(cluster_queries), 2),
        })

    for c in clusters:
        c["cluster_label"] = label_cluster(c["queries"])
        c["intent"] = classify_intent(c["queries"])

    cluster_df = pd.DataFrame(clusters)

    page_texts = (pages_df["title"] + ". " + pages_df["meta_description"] + ". " + pages_df["h1"]).tolist()
    page_vectors = np.array(get_embeddings(page_texts))

    best_pages, best_scores, mismatches = [], [], []
    for c in clusters:
        idx = [all_queries.index(q) for q in c["queries"]]
        centroid = np.mean([query_vectors[i] for i in idx], axis=0)
        sims = [cosine_similarity(centroid, pv) for pv in page_vectors]
        best_idx = int(np.argmax(sims))
        best_pages.append(pages_df.iloc[best_idx]["url"])
        best_scores.append(round(sims[best_idx], 3))
        is_guidance_or_output = c["intent"] in {"guidance-seeking", "output-seeking"}
        matched_a_product_page = "/products/" in pages_df.iloc[best_idx]["url"]
        mismatches.append(is_guidance_or_output and matched_a_product_page)

    cluster_df["best_page"] = best_pages
    cluster_df["best_page_score"] = best_scores
    cluster_df["intent_mismatch"] = mismatches

    threshold = float(cluster_df["best_page_score"].median())
    cluster_df["gap_flag"] = (cluster_df["best_page_score"] < threshold) | cluster_df["intent_mismatch"]
    cluster_df["threshold_used"] = threshold
    return cluster_df.sort_values("impressions", ascending=False).reset_index(drop=True)


perf_df = load_csv("performance_by_query.csv")
pages_df = load_csv("site_pages.csv")
gap_table = build_gap_table(perf_df, pages_df)
gap_table[["cluster_id", "cluster_label", "impressions", "intent", "best_page", "best_page_score", "gap_flag"]]

## Picking the gap to brief

The single highest-demand row flagged as a gap -- the one worth a full brief, rather than a lighter recommendation.

In [ ]:
flagged = gap_table[gap_table["gap_flag"]].sort_values("impressions", ascending=False)
if flagged.empty:
    raise SystemExit("No cluster in this run was flagged as a gap -- nothing to brief. Try lowering MIN_CLUSTER_SIZE or check the threshold logic above.")

top_gap = flagged.iloc[0].to_dict()
print(f"Briefing cluster {top_gap['cluster_id']}: \"{top_gap['cluster_label']}\"")
print(f"  {top_gap['impressions']} impressions, intent={top_gap['intent']}, score={top_gap['best_page_score']:.3f} (threshold {top_gap['threshold_used']:.3f})")
print(f"  currently best matched to: {top_gap['best_page']}")
print(f"  queries: {top_gap['queries']}")

## What a landing page brief needs

Same discipline the overview described for intent labeling: the taxonomy gets written down before anything is labeled, and here the brief's required fields get written down before anything is generated. A brief with no fixed shape is unreviewable and unscorable -- there's nothing consistent for a reviewer, or the evaluation step below, to check across more than one of them.

Ten required fields: the question this page targets, the real queries it targets, its intent, why the current best-matching page falls short (naming that page directly), a recommended title, meta description, and H1, a section outline, the existing pages it should link to (restricted to pages that actually exist), and a note on length and format informed by the cluster's own query shape -- the same word-count and question-form stats the overview tied to AI Overview appearance rates.

In [ ]:
import json

BRIEF_FIELDS = """
Return a JSON object with exactly these fields:
- "target_question": the question this page needs to answer (string)
- "target_queries": 3-6 of the real queries this page should target (list of strings, must
  be drawn from the queries given below, never invented)
- "stated_intent": the intent this page is being built for (string, one of: knowledge-seeking,
  guidance-seeking, output-seeking, navigational, transactional)
- "why_a_gap": 1-2 sentences on why the current best-matching page does not adequately serve
  this question -- name that page's URL directly (string)
- "recommended_title": a page title (string)
- "recommended_meta_description": a meta description under 160 characters (string)
- "recommended_h1": an H1 (string)
- "section_outline": 3-6 H2-level section headings for the page (list of strings)
- "internal_links": URLs of existing pages this new page should link to -- only URLs from
  the existing-pages list given below, never invented ones (list of strings)
- "notes_on_length_and_format": 1-2 sentences on how long and in what format this content
  should be, informed by the query shape stats given below (string)
"""


def parse_json_response(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:]
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        return {"parse_error": str(e), "raw_text": text}

### The guardrail

`draft_brief` refuses to run on a cluster that isn't flagged as a gap, checked in code before any model call happens -- not left for the model to notice on its own. This is the same instinct Session 5 applied to a model-written Cypher query: check what you can check deterministically before trusting a generation step to police itself.

In [ ]:
def draft_brief(gap_row: dict, pages_df: pd.DataFrame) -> dict:
    """Drafts a landing page brief for one gap-table row. Refuses rows not flagged as a gap."""
    if not gap_row["gap_flag"]:
        return {
            "refused": True,
            "reason": (f"cluster {gap_row['cluster_id']} is not flagged as a gap "
                       f"(score {gap_row['best_page_score']:.3f} against threshold "
                       f"{gap_row['threshold_used']:.3f}) -- nothing to brief."),
        }

    existing_pages = "\n".join(f"- {r.url} -- {r.title}" for r in pages_df.itertuples())
    prompt = f"""Write a landing page brief for a content team, based on real search data.

Target question (this cluster's label): {gap_row['cluster_label']}
Real queries in this cluster: {gap_row['queries']}
Intent (from the written taxonomy): {gap_row['intent']}
Current best-matching page: {gap_row['best_page']} (similarity score {gap_row['best_page_score']:.3f}, threshold {gap_row['threshold_used']:.3f})
Query shape: mean word count {gap_row['mean_word_count']}, {gap_row['pct_question_form']:.0%} of queries open with a question word

Existing pages on the site (for internal linking -- do not invent URLs not on this list):
{existing_pages}

{BRIEF_FIELDS}

Return only the JSON object, no other text."""
    return parse_json_response(call_llm(prompt))


brief = draft_brief(top_gap, pages_df)
brief

### Checking the guardrail actually holds

Run the same function against a cluster that *is* adequately served, and confirm it refuses before spending a single call on it.

In [ ]:
adequately_served = gap_table[~gap_table["gap_flag"]]
if adequately_served.empty:
    print("Every cluster in this run's gap table is flagged as a gap -- nothing to test the guardrail against. Try a stricter threshold in build_gap_table().")
else:
    non_gap = adequately_served.iloc[0].to_dict()
    print(draft_brief(non_gap, pages_df))

## Scoring the brief with the Week 3 template

Session 2's evaluation template compares predictions against known correct answers and reports where they disagree. Reusing it here means reframing brief quality as a set of pass/fail checks, each one a small classification task: a **true label** and a **predicted label** for the same yes/no question, for the same brief.

The predicted label comes from an **LLM judge** -- the model itself, asked to review the brief against the same real data it was supposed to be grounded in, and answer PASS or FAIL. The true label, for four of the five checks below, comes from **code that recomputes the same fact directly** -- does this URL actually appear in the page list, does this query actually appear in the cluster -- rather than from a second round of model judgment standing in for a human. That split matters for the same reason the SIGIR 2025 comparison in the overview did: a model asked to judge its own kind of output tends toward the same recall-favoring, precision-costing behavior as a model asked to classify one, so it's worth knowing, checkably, when the judge's PASS disagrees with what the data actually shows.

One of the five checks, whether the section outline is genuinely specific rather than generic filler, has no such objective check available -- judging that requires reading the actual text, so it's scored by the judge alone and flagged at the end as the one criterion this method can't check without a human doing exactly that.

In [ ]:
def check_cites_real_queries(brief: dict, gap_row: dict, pages_df: pd.DataFrame) -> str:
    cited = brief.get("target_queries")
    if not isinstance(cited, list) or not cited:
        return "Fail"
    real = {q.lower() for q in gap_row["queries"]}
    matched = sum(isinstance(q, str) and q.lower() in real for q in cited)
    return "Pass" if matched / len(cited) >= 0.5 else "Fail"


def check_correct_intent_named(brief: dict, gap_row: dict, pages_df: pd.DataFrame) -> str:
    stated = str(brief.get("stated_intent", "")).lower()
    return "Pass" if gap_row["intent"].lower() in stated else "Fail"


def check_real_internal_links_only(brief: dict, gap_row: dict, pages_df: pd.DataFrame) -> str:
    links = brief.get("internal_links")
    if not isinstance(links, list) or not links:
        return "Fail"
    real_urls = set(pages_df["url"])
    return "Pass" if all(link in real_urls for link in links) else "Fail"


def check_names_real_best_matching_page(brief: dict, gap_row: dict, pages_df: pd.DataFrame) -> str:
    reasoning = str(brief.get("why_a_gap", ""))
    return "Pass" if gap_row["best_page"] in reasoning else "Fail"


OBJECTIVE_CHECKS = {
    "cites_real_queries": check_cites_real_queries,
    "correct_intent_named": check_correct_intent_named,
    "real_internal_links_only": check_real_internal_links_only,
    "names_real_best_matching_page": check_names_real_best_matching_page,
}

CRITERION_QUESTIONS = {
    "cites_real_queries": "Do the brief's target_queries actually appear among the real cluster queries listed below (not invented)?",
    "correct_intent_named": "Does the brief's stated_intent match the real cluster intent given below?",
    "real_internal_links_only": "Are all of the brief's internal_links URLs ones that actually appear in the real existing-pages list below?",
    "names_real_best_matching_page": "Does the brief's why_a_gap reasoning correctly name the real best-matching page given below?",
    "concrete_section_outline": "Is the brief's section_outline specific and buildable (each heading names a real sub-topic), rather than generic filler that could apply to any page on any topic?",
}


def llm_judge(brief: dict, gap_row: dict, pages_df: pd.DataFrame, criterion: str) -> str:
    existing_pages = "\n".join(f"- {r.url}" for r in pages_df.itertuples())
    prompt = f"""You are reviewing a landing page brief against the real data it was supposed to be grounded in.

Real cluster queries: {gap_row['queries']}
Real cluster intent: {gap_row['intent']}
Real best-matching page: {gap_row['best_page']}
Real existing pages:
{existing_pages}

Brief under review:
{json.dumps(brief, indent=2)}

Question: {CRITERION_QUESTIONS[criterion]}
Answer with exactly one word: PASS or FAIL."""
    answer = call_llm(prompt).strip().upper()
    return "Pass" if answer.startswith("PASS") else "Fail"

### Generating a sample to evaluate

One brief is one data point. The cell below regenerates the brief for the same gap several times -- the same real cluster, the same real pages, a fresh call each time -- and scores every regeneration against all five criteria, giving a small but real sample instead of a single anecdote.

In [ ]:
N_REGENERATIONS = 5

generated_briefs = []
eval_rows = []
for i in range(N_REGENERATIONS):
    b = draft_brief(top_gap, pages_df)
    generated_briefs.append(b)

    for criterion, check_fn in OBJECTIVE_CHECKS.items():
        eval_rows.append({
            "brief_id": i, "criterion": criterion,
            "true_label": check_fn(b, top_gap, pages_df),
            "predicted_label": llm_judge(b, top_gap, pages_df, criterion),
        })
    # concrete_section_outline has no objective check -- judge-only, not part of eval_df below.

eval_df = pd.DataFrame(eval_rows)
eval_df

### Reusing `evaluate()`

Identical to the function built up across Session 2's evaluation template ("Week 3 template") -- copied here since notebooks in this course don't import from each other; see that notebook for how each piece of it was assembled and checked against the harmonic-mean formula by hand.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate(y_true, y_pred, labels=None, title="Evaluation Results"):
    """
    Evaluate classification predictions against ground truth.
    Works on ANY labeled classification task -- not just this dataset.

    y_true, y_pred : lists or pandas Series of labels (same length, same order)
    labels         : ordered list of all possible category names (optional --
                      inferred from the data if not given)
    """
    observed = set(y_true) | set(y_pred)
    if labels is None:
        labels = sorted(observed)
    else:
        labels = list(labels) + sorted(observed - set(labels))

    accuracy = accuracy_score(y_true, y_pred)
    print(f"{title}")
    print("=" * len(title))
    print(f"Overall accuracy: {accuracy:.1%}\n")

    print("Per-category breakdown (precision / recall / F1):")
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.title(title)
    plt.tight_layout()
    plt.show()

    return {"accuracy": accuracy, "confusion_matrix": cm}


overall = evaluate(eval_df["true_label"], eval_df["predicted_label"], labels=["Pass", "Fail"],
                    title="Brief rubric: judge's predicted label vs. the objective true label")

### Breaking it down by criterion

`evaluate()` on the whole table answers one question: overall, does the judge's PASS mean what it says. Grouping by criterion answers a sharper one -- which specific fact the pipeline gets wrong, and whether the judge catches it when it does.

In [ ]:
for criterion in OBJECTIVE_CHECKS:
    subset = eval_df[eval_df["criterion"] == criterion]
    objective_pass_rate = (subset["true_label"] == "Pass").mean()
    print(f"\n### {criterion}")
    print(f"objective pass rate across {len(subset)} regenerations: {objective_pass_rate:.0%}")
    evaluate(subset["true_label"], subset["predicted_label"], labels=["Pass", "Fail"], title=criterion)

### Which part of the pipeline produced the errors

Read your own numbers above rather than taking a description of them on faith. Two different kinds of failure are visible in this breakdown, and they trace to two different places:

A low **objective pass rate** on a criterion (regardless of what the judge says) means the brief-writing step itself is failing to carry forward what it was given -- it had the real queries, the real intent, the real best-matching page in its prompt, and still produced a brief that doesn't reflect them. That's a **generation-stage** failure, and it's only cleanly attributable to generation here because this notebook controls the gap table and can assert it's correct; in production, ruling out an upstream clustering or mapping error first is a real, separate step, not something to skip.

A gap between the **objective true label and the judge's predicted label** on a criterion where the objective pass rate is high means something different -- the brief is actually fine, but the judge doesn't reliably confirm it, or the reverse: the objective check fails but the judge says PASS anyway. That second pattern is the more concerning one, and it's the same asymmetry the SIGIR 2025 comparison in the overview described: a model judging its own kind of output tends to favor a generous PASS over a strict one, which is exactly why the true label above comes from recomputing the fact in code wherever that's possible, rather than from a second round of model opinion.

### What this method can't check

`concrete_section_outline` was left out of `eval_df` on purpose -- whether an outline is genuinely specific or generic filler isn't a fact any of the four checks above can recompute from data; it needs a person reading the actual headings. That's the point in this pipeline where automated scoring stops and a human's read is the only check available, the same point the previous notebook named for the agent's own output. The section outlines from every regeneration are printed below so that read can actually happen here, not just be gestured at.

In [ ]:
for i, b in enumerate(generated_briefs):
    print(f"--- brief {i} ---")
    print(b.get("section_outline", "(no section_outline field returned)"))
    print()

## What this costs

**In model calls**: rebuilding the gap table costs the same as the previous notebook (roughly 20 embedding and labeling calls). Each regenerated brief is one call; each criterion check against it is one more judge call, so five regenerations across four objectively-checkable criteria plus the one judge-only criterion is `5 x (1 + 5) = 30` calls for the evaluation section alone. None of this is expensive on a lightweight model -- Session 1's token cost guide covers the exact method, and pricing changes, so check [AI Studio](https://aistudio.google.com) rather than trusting a number written on a fixed date.

**In human work**: the gap table's threshold here still uses the placeholder shortcut, not a real labeled sample -- swap in the previous notebook's actual three-step procedure before trusting which clusters get briefed at all. And `concrete_section_outline`, as covered above, needs a human read every time; nothing in this notebook scores it for you.

In [ ]:
token_count = client.models.count_tokens(model=MODEL_NAME, contents=json.dumps(brief))
print(f"One judge call's brief-and-context payload is roughly {token_count.total_tokens} input tokens.")
print("Check current per-token pricing at https://aistudio.google.com before scaling this up.")

## What you built

A single gap turned into a full, structured brief, regenerated several times and scored against a rubric that mostly checks itself: four of five criteria compare an LLM judge's opinion against a fact recomputed directly from the same real data the brief was supposed to use, and the fifth is named, explicitly, as the one a person still has to read. That split is the actual takeaway from this week, not just this notebook -- automating a judgment all the way through is possible exactly where the judgment reduces to a checkable fact, and not possible where it doesn't, and knowing which is which for a given pipeline is worth more than trusting either a model's confidence or a human's spot-check alone.

The evaluation here rests on one gap and five regenerations, which is a demonstration of the method, not a benchmark of the pipeline's typical reliability -- the honest generalization is narrower than "the brief-writing step is this accurate," and a real rollout would want this same table run across every flagged gap, not just the highest-demand one, before trusting the pass rates as a property of the system rather than of this one question.

**This closes the course.** Five weeks built the pieces this pipeline draws on: evaluation metrics and human-in-the-loop review from Session 2, the compliance instincts that show up here as the guardrail refusing to brief a non-gap, the comfort with synthetic data that made this week's dataset possible to build honestly, and the knowledge-graph week's own groundedness check, which is exactly the question this notebook keeps asking of a landing page brief instead of a graph answer.

**Questions?** Post in the Circle community.